# Forest Impact Simulator

This notebook simulates the environmental impact of forest management activities including planting and clear-cutting operations.

## Features
- **Input Options**: Upload CSV or manually input land plot data
- **Management Modes**: Planting and clear-cutting simulations
- **Impact Calculations**: Carbon sequestration, biodiversity, forest resilience, water retention, air quality
- **Visualizations**: Charts and plots for environmental impacts
- **Export**: Results as CSV and JSON files

## Getting Started
1. Install required packages: `pip install -r requirements.txt`
2. Run all cells to initialize the simulator
3. Input your data or upload a CSV file
4. Select management mode and run simulation
5. View results and export data


## 1. Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")


## 2. Forest Impact Calculator Class


In [ ]:
class ForestImpactCalculator:
    """
    A comprehensive forest impact calculator for simulating environmental effects
    of forest management activities including planting and clear-cutting.
    """
    
    def __init__(self):
        # Carbon sequestration rates by tree type (kg CO2/year/tree)
        self.carbon_rates = {
            'Oak': 22.0,
            'Pine': 18.5,
            'Maple': 20.0,
            'Birch': 16.0,
            'Spruce': 19.0,
            'Cedar': 21.0,
            'Redwood': 25.0,
            'Eucalyptus': 30.0,
            'Mixed': 20.0,
            'Other': 18.0
        }
        
        # Biodiversity scores by tree type (0-100 scale)
        self.biodiversity_scores = {
            'Oak': 85,
            'Pine': 70,
            'Maple': 80,
            'Birch': 75,
            'Spruce': 65,
            'Cedar': 90,
            'Redwood': 95,
            'Eucalyptus': 60,
            'Mixed': 85,
            'Other': 70
        }
        
        # Forest resilience factors
        self.resilience_factors = {
            'Oak': 0.85,
            'Pine': 0.75,
            'Maple': 0.80,
            'Birch': 0.70,
            'Spruce': 0.78,
            'Cedar': 0.90,
            'Redwood': 0.95,
            'Eucalyptus': 0.65,
            'Mixed': 0.85,
            'Other': 0.75
        }
        
        # Water retention capacity (liters/m²/year)
        self.water_retention = {
            'Oak': 1200,
            'Pine': 1000,
            'Maple': 1100,
            'Birch': 950,
            'Spruce': 1050,
            'Cedar': 1300,
            'Redwood': 1400,
            'Eucalyptus': 800,
            'Mixed': 1150,
            'Other': 1000
        }
        
        # Air quality improvement (particulate reduction %)
        self.air_quality_improvement = {
            'Oak': 15,
            'Pine': 12,
            'Maple': 14,
            'Birch': 11,
            'Spruce': 13,
            'Cedar': 16,
            'Redwood': 18,
            'Eucalyptus': 10,
            'Mixed': 14,
            'Other': 12
        }
    
    def calculate_plot_impacts(self, plot_data, mode='planting'):
        """
        Calculate environmental impacts for a single plot.
        
        Args:
            plot_data (dict): Plot information including area, tree_type, etc.
            mode (str): 'planting' or 'clear-cutting'
            
        Returns:
            dict: Calculated impacts for the plot
        """
        area = plot_data.get('area', 0)  # in hectares
        tree_type = plot_data.get('tree_type', 'Other')
        tree_count = plot_data.get('tree_count', 0)
        tree_density = plot_data.get('tree_density', 0)  # trees per hectare
        
        # Calculate tree count if not provided
        if tree_count == 0 and tree_density > 0:
            tree_count = area * tree_density
        elif tree_count == 0 and area > 0:
            # Default density based on tree type
            default_density = {
                'Oak': 200, 'Pine': 300, 'Maple': 250, 'Birch': 280,
                'Spruce': 350, 'Cedar': 180, 'Redwood': 150, 'Eucalyptus': 400,
                'Mixed': 250, 'Other': 250
            }
            tree_count = area * default_density.get(tree_type, 250)
        
        # Calculate impacts based on mode
        multiplier = 1 if mode == 'planting' else -1
        
        # Carbon sequestration (kg CO2/year)
        carbon_per_tree = self.carbon_rates.get(tree_type, 18.0)
        carbon_sequestration = tree_count * carbon_per_tree * multiplier
        
        # Biodiversity impact (weighted by area and tree count)
        biodiversity_score = self.biodiversity_scores.get(tree_type, 70)
        biodiversity_impact = (area * biodiversity_score * 0.1 + 
                              tree_count * biodiversity_score * 0.01) * multiplier
        
        # Forest resilience
        resilience_factor = self.resilience_factors.get(tree_type, 0.75)
        forest_resilience = area * resilience_factor * 10 * multiplier
        
        # Water retention (liters/year)
        water_per_m2 = self.water_retention.get(tree_type, 1000)
        water_retention = area * 10000 * water_per_m2 * multiplier  # Convert hectares to m²
        
        # Air quality improvement (%)
        air_quality = self.air_quality_improvement.get(tree_type, 12)
        air_quality_improvement = (area * air_quality * 0.1 + 
                                  tree_count * air_quality * 0.001) * multiplier
        
        return {
            'plot_id': plot_data.get('plot_id', 'Unknown'),
            'area': area,
            'tree_type': tree_type,
            'tree_count': tree_count,
            'carbon_sequestration': carbon_sequestration,
            'biodiversity_impact': biodiversity_impact,
            'forest_resilience': forest_resilience,
            'water_retention': water_retention,
            'air_quality_improvement': air_quality_improvement
        }
    
    def calculate_total_impacts(self, plots_data, mode='planting'):
        """
        Calculate total impacts across all plots.
        
        Args:
            plots_data (list): List of plot dictionaries
            mode (str): 'planting' or 'clear-cutting'
            
        Returns:
            dict: Total and average impacts
        """
        plot_results = []
        
        for plot in plots_data:
            result = self.calculate_plot_impacts(plot, mode)
            plot_results.append(result)
        
        # Calculate totals and averages
        total_carbon = sum(r['carbon_sequestration'] for r in plot_results)
        total_biodiversity = sum(r['biodiversity_impact'] for r in plot_results)
        total_resilience = sum(r['forest_resilience'] for r in plot_results)
        total_water = sum(r['water_retention'] for r in plot_results)
        total_air_quality = sum(r['air_quality_improvement'] for r in plot_results)
        
        num_plots = len(plot_results)
        
        return {
            'plot_results': plot_results,
            'total_carbon': total_carbon,
            'average_biodiversity': total_biodiversity / num_plots if num_plots > 0 else 0,
            'average_resilience': total_resilience / num_plots if num_plots > 0 else 0,
            'total_water_retention': total_water,
            'total_air_quality_improvement': total_air_quality,
            'num_plots': num_plots
        }

# Initialize the calculator
calculator = ForestImpactCalculator()
print("✅ Forest Impact Calculator initialized!")


## 3. Data Input and Management

This section provides options for inputting forest plot data either through CSV upload or manual entry.


In [ ]:
# Global variables to store data
plots_data = []
simulation_results = None

def create_sample_data():
    """Create sample forest plot data for demonstration."""
    sample_plots = [
        {
            'plot_id': 'Plot_001',
            'area': 5.2,  # hectares
            'tree_type': 'Oak',
            'tree_count': 1040,
            'tree_density': 200,
            'latitude': 45.5231,
            'longitude': -122.6765,
            'soil_type': 'Loamy',
            'climate_zone': 'Temperate'
        },
        {
            'plot_id': 'Plot_002',
            'area': 3.8,
            'tree_type': 'Pine',
            'tree_count': 1140,
            'tree_density': 300,
            'latitude': 45.5240,
            'longitude': -122.6770,
            'soil_type': 'Sandy',
            'climate_zone': 'Temperate'
        },
        {
            'plot_id': 'Plot_003',
            'area': 7.1,
            'tree_type': 'Mixed',
            'tree_count': 1775,
            'tree_density': 250,
            'latitude': 45.5220,
            'longitude': -122.6750,
            'soil_type': 'Clay',
            'climate_zone': 'Temperate'
        }
    ]
    return sample_plots

def load_csv_data(file_path):
    """
    Load forest plot data from CSV file.
    
    Expected CSV columns:
    - plot_id: Unique identifier for each plot
    - area: Area in hectares
    - tree_type: Type of trees (Oak, Pine, Maple, etc.)
    - tree_count: Number of trees (optional)
    - tree_density: Trees per hectare (optional)
    - latitude: GPS latitude (optional)
    - longitude: GPS longitude (optional)
    - soil_type: Soil classification (optional)
    - climate_zone: Climate zone (optional)
    """
    try:
        df = pd.read_csv(file_path)
        
        # Convert DataFrame to list of dictionaries
        plots = []
        for _, row in df.iterrows():
            plot = {
                'plot_id': str(row.get('plot_id', f'Plot_{len(plots)+1:03d}')),
                'area': float(row.get('area', 0)),
                'tree_type': str(row.get('tree_type', 'Other')),
                'tree_count': int(row.get('tree_count', 0)),
                'tree_density': int(row.get('tree_density', 0)),
                'latitude': float(row.get('latitude', 0)),
                'longitude': float(row.get('longitude', 0)),
                'soil_type': str(row.get('soil_type', 'Unknown')),
                'climate_zone': str(row.get('climate_zone', 'Unknown'))
            }
            plots.append(plot)
        
        print(f"✅ Successfully loaded {len(plots)} plots from CSV file")
        return plots
        
    except Exception as e:
        print(f"❌ Error loading CSV file: {str(e)}")
        return []

def display_plots_summary(plots):
    """Display a summary of loaded plots data."""
    if not plots:
        print("No plots data available.")
        return
    
    df = pd.DataFrame(plots)
    print(f"\n📊 Forest Plots Summary ({len(plots)} plots)")
    print("=" * 50)
    
    # Display basic statistics
    print(f"Total Area: {df['area'].sum():.2f} hectares")
    print(f"Average Area: {df['area'].mean():.2f} hectares")
    print(f"Total Trees: {df['tree_count'].sum():,}")
    print(f"Average Trees per Plot: {df['tree_count'].mean():.0f}")
    
    # Tree type distribution
    print(f"\nTree Type Distribution:")
    tree_counts = df['tree_type'].value_counts()
    for tree_type, count in tree_counts.items():
        print(f"  {tree_type}: {count} plots")
    
    # Display first few rows
    print(f"\nFirst 5 plots:")
    display(df.head())

print("✅ Data management functions loaded!")


### 3.1 Load Sample Data or Upload CSV

Choose one of the options below to load forest plot data:


In [ ]:
# Option 1: Load sample data for demonstration
def load_sample_data():
    global plots_data
    plots_data = create_sample_data()
    display_plots_summary(plots_data)

# Option 2: Load data from CSV file
def load_csv_file():
    file_path = input("Enter the path to your CSV file: ")
    global plots_data
    plots_data = load_csv_data(file_path)
    if plots_data:
        display_plots_summary(plots_data)

# Option 3: Manual data entry
def manual_data_entry():
    print("Manual data entry - Add plots one by one")
    print("Enter 'done' when finished adding plots")
    
    global plots_data
    plots_data = []
    
    while True:
        plot_id = input("Enter plot ID (or 'done' to finish): ")
        if plot_id.lower() == 'done':
            break
            
        try:
            area = float(input("Enter area in hectares: "))
            tree_type = input("Enter tree type (Oak, Pine, Maple, etc.): ")
            tree_count = int(input("Enter tree count (0 if unknown): "))
            tree_density = int(input("Enter tree density per hectare (0 if unknown): "))
            latitude = float(input("Enter latitude (0 if unknown): "))
            longitude = float(input("Enter longitude (0 if unknown): "))
            
            plot = {
                'plot_id': plot_id,
                'area': area,
                'tree_type': tree_type,
                'tree_count': tree_count,
                'tree_density': tree_density,
                'latitude': latitude,
                'longitude': longitude,
                'soil_type': 'Unknown',
                'climate_zone': 'Unknown'
            }
            plots_data.append(plot)
            print(f"✅ Added plot {plot_id}")
            
        except ValueError:
            print("❌ Invalid input. Please try again.")
    
    if plots_data:
        display_plots_summary(plots_data)

# Interactive buttons for data loading
print("Choose how to load your forest plot data:")
print("1. Load sample data (for demonstration)")
print("2. Load from CSV file")
print("3. Manual data entry")
print("\nRun the appropriate function below:")


## 4. Simulation Configuration

Configure simulation parameters including management mode and simulation years.


In [ ]:
# Simulation configuration
simulation_config = {
    'mode': 'planting',  # 'planting' or 'clear-cutting'
    'simulation_years': 10,
    'location': {
        'latitude': 45.5231,
        'longitude': -122.6765,
        'name': 'Portland, Oregon'
    },
    'environmental_data': {
        'soil_type': 'Loamy',
        'climate_zone': 'Temperate',
        'precipitation': 1000,  # mm/year
        'temperature': 12.5  # °C average
    }
}

def configure_simulation():
    """Configure simulation parameters interactively."""
    global simulation_config
    
    print("🌲 Forest Impact Simulation Configuration")
    print("=" * 50)
    
    # Management mode
    mode = input("Enter management mode (planting/clear-cutting): ").lower()
    if mode in ['planting', 'clear-cutting']:
        simulation_config['mode'] = mode
    else:
        print("Invalid mode. Using default: planting")
        simulation_config['mode'] = 'planting'
    
    # Simulation years
    try:
        years = int(input("Enter simulation years (default 10): ") or "10")
        simulation_config['simulation_years'] = years
    except ValueError:
        simulation_config['simulation_years'] = 10
    
    # Location
    try:
        lat = float(input("Enter latitude (default 45.5231): ") or "45.5231")
        lon = float(input("Enter longitude (default -122.6765): ") or "-122.6765")
        location_name = input("Enter location name (default 'Portland, Oregon'): ") or "Portland, Oregon"
        
        simulation_config['location'] = {
            'latitude': lat,
            'longitude': lon,
            'name': location_name
        }
    except ValueError:
        print("Invalid coordinates. Using defaults.")
    
    print(f"\n✅ Configuration updated:")
    print(f"   Mode: {simulation_config['mode']}")
    print(f"   Years: {simulation_config['simulation_years']}")
    print(f"   Location: {simulation_config['location']['name']}")
    print(f"   Coordinates: {simulation_config['location']['latitude']}, {simulation_config['location']['longitude']}")

# Run configuration
configure_simulation()
